In [ ]:
import cv2 as cv              # OpenCV计算机视觉库
import numpy as np            # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        plt.imshow(img)
    plt.show()

# 1.角点检测

## 1.1Harris角点检测

In [ ]:
img = cv.imread("pic/shudu.png")  # 读取数独图片用于Harris角点检测
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)  # 将原始图像灰度化，角点检测基于灰度图
show(gray)

In [ ]:
# Harris角点检测：基于图像梯度的自相关矩阵特征值分析
blockSize = 2  # 检测窗口大小（邻域尺寸）
ksize = 3      # Sobel算子的卷积核大小
k = 0.01       # Harris响应函数中的自由参数，通常取0.04~0.06

dst = cv.cornerHarris(gray, blockSize, ksize, k)  # 返回每个像素的Harris响应值R
print(dst)

In [ ]:
# 将Harris角点检测的结果画在图像上，结果显示为红色
# 大于阈值0.01 * dst.max()时才显示出来，否则不显示
img[dst > 0.01 * dst.max()] = [0, 0, 255]  # BGR格式，[0,0,255]为红色

show(img)

## 1.2 Shi-Tomasi角点

In [ ]:
img = cv.imread("pic/corner.jpeg")  # 读取建筑图片用于Shi-Tomasi角点检测
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)  # 将原始图像灰度化
show(gray)

In [ ]:
# Shi-Tomasi角点检测：改进的Harris方法，使用最小特征值判断角点质量
maxCorners = 1000      # 想要检测的最大角点数目
qualityLevel = 0.01    # 角点质量水平（相对于最佳角点的质量比值）
minDistance = 20        # 两个角点之间的最小欧式距离，距离越大检测到的角点越少

# goodFeaturesToTrack：Shi-Tomasi角点检测的核心函数
corners = cv.goodFeaturesToTrack(gray, maxCorners, qualityLevel, minDistance)

In [ ]:
# 在原图上绘制检测到的Shi-Tomasi角点
corners = np.int32(corners)  # 将浮点坐标转为整数
for i in corners:
    x, y = i.ravel()  # ravel将二维数组展平为一维，提取(x,y)坐标
    cv.circle(img, (x,y), 3, (255,0,0), -1)  # 绘制蓝色实心圆点，半径3，-1表示填充

show(img)

## 1.3 Fast角点

In [ ]:
img = cv.imread("pic/corner.jpeg")  # 读取建筑图片用于FAST角点检测
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)  # 将原始图像灰度化
show(gray)

In [ ]:
# 初始化FAST角点检测器：通过比较像素周围圆形邻域的亮度快速检测角点
threshold = 7                  # 亮度差阈值，中心像素与周围像素亮度差超过此值才考虑为角点
nonmaxSuppression = True       # 使用非极大值抑制，避免在角点附近重复检测
type = cv.FAST_FEATURE_DETECTOR_TYPE_7_12  # 邻域类型：圆周16个像素中连续12个亮或暗

fast = cv.FastFeatureDetector_create(threshold, nonmaxSuppression, type)

In [ ]:
# 使用FAST检测器查找关键点并在图上绘制
kp = fast.detect(img, None)  # detect返回关键点列表（KeyPoint对象）
img2 = cv.drawKeypoints(gray, kp, None, color=(255,0,0))  # 在灰度图上绘制蓝色关键点
show(img2)